In [1]:
import json
import pandas as pd
from functools import reduce

In [2]:
file_path = "../../data/conv-trs/ecir-2026/human-eval/surveys_collection_v2.json"

with open(file_path, 'r') as f:
    data = json.load(f)
data = [person for person in data if person["prolificId"] in(["Ewald_cleaned", "Wolfgang"])]
len(data)

2

In [3]:
data[0]["prolificId"] = "Ewald"

In [4]:
# Build a dictionary: query_id -> {query, annotator1_dim1_score, annotator1_dim1_not_sure, ...}
query_data = {}

for annotator in data:
    prolific_id = annotator['prolificId']
    print(f"Processing: {prolific_id}")

    responses = annotator['answers']['ranking-eval']

    for r in responses:
        query_id = r['query_id']

        # Initialize query entry if not exists
        if query_id not in query_data:
            query_data[query_id] = {
                'query_id': query_id,
                'query': r.get('query', '')
            }

        # Extract all dimension fields (those with value, label, not_sure keys)
        for field_name, field_data in r.items():
            if isinstance(field_data, dict) and 'value' in field_data:
                # Clean up dimension name - remove prefixes and suffixes
                dim_name = field_name
                # Remove dim_ prefix if present
                if dim_name.startswith('dim_'):
                    dim_name = dim_name[4:]  # Remove 'dim_'
                # Remove various qid patterns
                dim_name = dim_name.replace('_qid_0', '').replace('_qid_', '').replace('_qid', '')
                # Remove _match suffix
                if dim_name.endswith('_match'):
                    dim_name = dim_name[:-6]  # Remove '_match'

                # Add columns for this annotator's ratings
                query_data[query_id][f'{prolific_id}_{dim_name}_score'] = field_data.get('value')
                # query_data[query_id][f'{prolific_id}_{dim_name}_label'] = field_data.get('label')
                query_data[query_id][f'{prolific_id}_{dim_name}_not_sure'] = field_data.get('not_sure', False)

# Convert to DataFrame
merged_df = pd.DataFrame.from_dict(query_data, orient='index').reset_index(drop=True)

# Sort columns: query_id, query, then all annotator columns
base_cols = ['query_id', 'query']
other_cols = sorted([c for c in merged_df.columns if c not in base_cols])
merged_df = merged_df[base_cols + other_cols]
merged_df.drop(columns=["Ewald_sustainability_not_sure_not_sure",\
                        "Ewald_sustainability_not_sure_score", \
                       "Wolfgang_diversity_not_sure_not_sure","Wolfgang_diversity_not_sure_score", \
                       "Wolfgang_popularity_mix_not_sure_not_sure", "Wolfgang_sustainability_not_sure_score",
                       "Wolfgang_sustainability_not_sure_not_sure", "Wolfgang_popularity_mix_not_sure_score"],\
              inplace = True)
print(f"\nShape: {merged_df.shape}")
print(f"Columns: {list(merged_df.columns)}")
merged_df.fillna(0, inplace=True)

merged_df.to_csv("../../data/conv-trs/ecir-2026/human-eval/survey_data_cleaned.csv", index=False)
og_merged_df = merged_df.copy()
merged_df

Processing: Ewald
Processing: Wolfgang

Shape: (101, 18)
Columns: ['query_id', 'query', 'Ewald_diversity_not_sure', 'Ewald_diversity_score', 'Ewald_popularity_mix_not_sure', 'Ewald_popularity_mix_score', 'Ewald_relevance_not_sure', 'Ewald_relevance_score', 'Ewald_sustainability_not_sure', 'Ewald_sustainability_score', 'Wolfgang_diversity_not_sure', 'Wolfgang_diversity_score', 'Wolfgang_popularity_mix_not_sure', 'Wolfgang_popularity_mix_score', 'Wolfgang_relevance_not_sure', 'Wolfgang_relevance_score', 'Wolfgang_sustainability_not_sure', 'Wolfgang_sustainability_score']


,query_id,query,Ewald_diversity_not_sure,Ewald_diversity_score,Ewald_popularity_mix_not_sure,Ewald_popularity_mix_score,Ewald_relevance_not_sure,Ewald_relevance_score,Ewald_sustainability_not_sure,Ewald_sustainability_score,Wolfgang_diversity_not_sure,Wolfgang_diversity_score,Wolfgang_popularity_mix_not_sure,Wolfgang_popularity_mix_score,Wolfgang_relevance_not_sure,Wolfgang_relevance_score,Wolfgang_sustainability_not_sure,Wolfgang_sustainability_score
0,c_p_68_pop_medium_sustainable,Budget-friendly European city break in April.,False,3.0,False,4.0,False,4.0,False,4.0,False,3,False,3,False,4,False,4
1,c_p_196_pop_high_sustainable,Cheap European city break in October with muse...,False,2.0,False,4.0,False,2.0,False,2.0,False,3,False,2,False,1,False,2
2,c_p_9_pop_medium_sustainable,Walkable European city break in June with inte...,False,4.0,False,4.0,False,5.0,False,4.0,False,3,False,3,False,1,False,3
3,c_p_1_pop_medium_easy,Suggest a mid-range European city break with i...,False,2.0,False,4.0,False,4.0,False,3.0,False,4,False,4,False,5,False,4
4,c_p_38_pop_low_sustainable,October trip to a less-touristy European coast...,False,4.0,False,4.0,False,1.0,False,4.0,False,3,False,3,False,3,False,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,c_p_97_pop_low_sustainable,"Suggest a low-popularity, walkable European ci...",False,3.0,False,3.0,False,3.0,False,3.0,False,2,False,3,False,3,False,4
97,c_p_2_pop_medium_hard,High budget European city break in July with i...,False,3.0,False,2.0,False,2.0,False,3.0,False,3,False,3,False,1,False,2
98,c_p_51_pop_high_sustainable,Low budget European city break with good air q...,False,3.0,False,4.0,False,2.0,False,3.0,False,3,False,3,False,3,False,3
99,c_p_51_pop_medium_sustainable,Low budget European city break in February wit...,False,3.0,False,3.0,False,3.0,False,3.0,False,3,False,3,False,2,False,2


In [5]:
merged_df.columns

Index(['query_id', 'query', 'Ewald_diversity_not_sure',
       'Ewald_diversity_score', 'Ewald_popularity_mix_not_sure',
       'Ewald_popularity_mix_score', 'Ewald_relevance_not_sure',
       'Ewald_relevance_score', 'Ewald_sustainability_not_sure',
       'Ewald_sustainability_score', 'Wolfgang_diversity_not_sure',
       'Wolfgang_diversity_score', 'Wolfgang_popularity_mix_not_sure',
       'Wolfgang_popularity_mix_score', 'Wolfgang_relevance_not_sure',
       'Wolfgang_relevance_score', 'Wolfgang_sustainability_not_sure',
       'Wolfgang_sustainability_score'],
      dtype='object')

In [6]:
SURVEY_LLM_SCORES_MAPPING = {
    1.0: 2,
    2.0: 1,
    3.0: 0,
    4.0: -1,
    5.0: -2
}
annotators = ["Ewald", "Wolfgang"]
dimensions = ["diversity", "relevance", "sustainability", "popularity_mix"]

In [7]:
og_merged_df.Wolfgang_sustainability_score.value_counts()

Wolfgang_sustainability_score
3    51
4    25
2    23
5     1
1     1
Name: count, dtype: int64

In [8]:

unsure_cols = []
for annotator in annotators:
    for dimension in dimensions:
        col_name = annotator + "_" + dimension +"_score"
        unsure_col = annotator + "_" + dimension +"_not_sure"
        merged_df.loc[merged_df[unsure_col] == False, col_name] = (
            merged_df.loc[merged_df[unsure_col] == False, col_name]
            .map(SURVEY_LLM_SCORES_MAPPING)
        )

        # Assign -3 where 'not_sure' == True
        merged_df.loc[merged_df[unsure_col] == True, col_name] = -3
        unsure_cols.append(unsure_col)


In [9]:
unsure_cols

['Ewald_diversity_not_sure',
 'Ewald_relevance_not_sure',
 'Ewald_sustainability_not_sure',
 'Ewald_popularity_mix_not_sure',
 'Wolfgang_diversity_not_sure',
 'Wolfgang_relevance_not_sure',
 'Wolfgang_sustainability_not_sure',
 'Wolfgang_popularity_mix_not_sure']

In [10]:
merged_df.Wolfgang_sustainability_not_sure.value_counts()

Wolfgang_sustainability_not_sure
False    78
True     23
Name: count, dtype: int64

In [11]:
merged_df.Wolfgang_sustainability_score.value_counts()

Wolfgang_sustainability_score
 0    28
-1    25
 1    23
-3    23
-2     1
 2     1
Name: count, dtype: int64

In [12]:
merged_df.isna().sum()

query_id                            0
query                               0
Ewald_diversity_not_sure            0
Ewald_diversity_score               1
Ewald_popularity_mix_not_sure       0
Ewald_popularity_mix_score          1
Ewald_relevance_not_sure            0
Ewald_relevance_score               1
Ewald_sustainability_not_sure       0
Ewald_sustainability_score          1
Wolfgang_diversity_not_sure         0
Wolfgang_diversity_score            0
Wolfgang_popularity_mix_not_sure    0
Wolfgang_popularity_mix_score       0
Wolfgang_relevance_not_sure         0
Wolfgang_relevance_score            0
Wolfgang_sustainability_not_sure    0
Wolfgang_sustainability_score       0
dtype: int64

In [13]:
merged_df.drop(columns = unsure_cols, inplace=True)
merged_df

,query_id,query,Ewald_diversity_score,Ewald_popularity_mix_score,Ewald_relevance_score,Ewald_sustainability_score,Wolfgang_diversity_score,Wolfgang_popularity_mix_score,Wolfgang_relevance_score,Wolfgang_sustainability_score
0,c_p_68_pop_medium_sustainable,Budget-friendly European city break in April.,0.0,-1.0,-1.0,-1.0,0,0,-1,-1
1,c_p_196_pop_high_sustainable,Cheap European city break in October with muse...,1.0,-1.0,1.0,1.0,0,1,2,1
2,c_p_9_pop_medium_sustainable,Walkable European city break in June with inte...,-1.0,-1.0,-2.0,-1.0,0,0,2,0
3,c_p_1_pop_medium_easy,Suggest a mid-range European city break with i...,1.0,-1.0,-1.0,0.0,-1,-1,-2,-1
4,c_p_38_pop_low_sustainable,October trip to a less-touristy European coast...,-1.0,-1.0,2.0,-1.0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
96,c_p_97_pop_low_sustainable,"Suggest a low-popularity, walkable European ci...",0.0,0.0,0.0,0.0,1,0,0,-1
97,c_p_2_pop_medium_hard,High budget European city break in July with i...,0.0,1.0,1.0,0.0,0,0,2,1
98,c_p_51_pop_high_sustainable,Low budget European city break with good air q...,0.0,-1.0,1.0,0.0,0,0,0,0
99,c_p_51_pop_medium_sustainable,Low budget European city break in February wit...,0.0,0.0,0.0,0.0,0,0,1,1


In [15]:
merged_df.isna().sum()

query_id                         0
query                            0
Ewald_diversity_score            1
Ewald_popularity_mix_score       1
Ewald_relevance_score            1
Ewald_sustainability_score       1
Wolfgang_diversity_score         0
Wolfgang_popularity_mix_score    0
Wolfgang_relevance_score         0
Wolfgang_sustainability_score    0
dtype: int64

In [14]:
merged_df.to_csv("../../data/conv-trs/ecir-2026/human-eval/survey_data_cleaned_mapped.csv", index=False)